In [1]:
"""
Test: compare naive restarted Arnoldi vs IRAM.
Run with:  JAX_ENABLE_X64=1 python test_iram.py
"""
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

from dynamiqs.steady_state.core.arnoldi_iram import arnoldi_iram_lm_jit
from dynamiqs.steady_state.core.arnoldi import arnoldi_restarted_lm_jit


def test(name, A, m, max_cycles=80, tol=1e-12):
    N = A.shape[0]
    evals_true = jnp.linalg.eigvals(A)
    true_eval = evals_true[jnp.argmax(jnp.abs(evals_true))]

    key = jax.random.PRNGKey(42)
    v0 = jax.random.normal(key, (N,), dtype=jnp.float64)
    v0 = v0.astype(jnp.complex128)
    matvec = lambda v: A @ v

    tn, _, infn = arnoldi_restarted_lm_jit(matvec, v0, m=m, max_cycles=max_cycles, tol=tol)
    ti, _, infi = arnoldi_iram_lm_jit(matvec, v0, m=m, k=1, max_cycles=max_cycles, tol=tol)

    print(f"\n{'=' * 60}")
    print(f"  {name}  (N={N}, m={m})")
    print(f"{'=' * 60}")
    print(f"  True λ_max  = {true_eval}")
    print(f"  Naive: λ={tn:.10f}  cycles={infn.n_cycles}  ritz_res={infn.ritz_residual:.2e}")
    print(f"  IRAM:  λ={ti:.10f}  cycles={infi.n_cycles}  ritz_res={infi.ritz_residual:.2e}")
    print(f"  Naive matvecs={infn.n_inner_iters}  IRAM matvecs={infi.n_inner_iters}")
    print(f"  Error naive={jnp.abs(tn - true_eval):.2e}  IRAM={jnp.abs(ti - true_eval):.2e}")


# 1. Diagonal
N = 100
test("Diagonal", jnp.diag(jnp.arange(1.0, N + 1)), m=15)

# 2. Random symmetric
key = jax.random.PRNGKey(123)
M = jax.random.normal(key, (200, 200), dtype=jnp.float64)
test("Symmetric", (M + M.T) / 2, m=20, tol=1e-10)

# 3. Non-symmetric
A = jax.random.normal(jax.random.PRNGKey(999), (150, 150), dtype=jnp.float64)
test("Non-symmetric", A, m=25, tol=1e-10)

# 4. JIT
print(f"\n{'=' * 60}")
print("  JIT compilation test")
print(f"{'=' * 60}")
A = jnp.diag(jnp.arange(1.0, 51.0))

@jax.jit
def run(v0):
    return arnoldi_iram_lm_jit(lambda v: A @ v, v0, m=10, k=1, max_cycles=30, tol=1e-10)

v0 = jax.random.normal(jax.random.PRNGKey(0), (50,))
v0 = v0.astype(jnp.complex128)
t, _, info = run(v0)

print(f"  JIT OK: λ={jnp.real(t):.10f} (expect 50), cycles={info.n_cycles}")


  Diagonal  (N=100, m=15)
  True λ_max  = (100+0j)
  Naive: λ=100.0000000000+0.0000000000j  cycles=13  ritz_res=4.10e-13
  IRAM:  λ=100.0000000000+0.0000000000j  cycles=13  ritz_res=4.13e-13
  Naive matvecs=195  IRAM matvecs=183
  Error naive=1.56e-13  IRAM=8.53e-14

  Symmetric  (N=200, m=20)
  True λ_max  = (19.90850298592405+0j)
  Naive: λ=19.9085029859-0.0000000000j  cycles=7  ritz_res=7.48e-13
  IRAM:  λ=19.9085029859-0.0000000000j  cycles=7  ritz_res=7.48e-13
  Naive matvecs=140  IRAM matvecs=134
  Error naive=2.49e-14  IRAM=1.07e-14

  Non-symmetric  (N=150, m=25)
  True λ_max  = (0.8613645250255717+12.490419235958985j)
  Naive: λ=0.8613645250-12.4904192363j  cycles=80  ritz_res=4.05e-09
  IRAM:  λ=0.8613645250-12.4904192363j  cycles=80  ritz_res=4.05e-09
  Naive matvecs=2000  IRAM matvecs=1921
  Error naive=2.50e+01  IRAM=2.50e+01

  JIT compilation test
  JIT OK: λ=50.0000000000 (expect 50), cycles=11


In [2]:
"""
Test: compare naive restarted Arnoldi vs IRAM.
Run with:  JAX_ENABLE_X64=1 python test_iram.py

The key: with nev=1, using k=1 makes IRAM degenerate to naive restart.
We need k >= 2 ("guard vectors") for the polynomial filter to help.
Default k = min(max(2*nev, nev+2), m-1) = 3 when nev=1.
"""
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)


from dynamiqs.steady_state.core.arnoldi_iram import arnoldi_iram_lm_jit
from dynamiqs.steady_state.core.arnoldi import arnoldi_restarted_lm_jit


def test(name, A, m, max_cycles=80, tol=1e-12):
    N = A.shape[0]
    evals_true = jnp.linalg.eigvals(A)
    true_eval = evals_true[jnp.argmax(jnp.abs(evals_true))]

    key = jax.random.PRNGKey(42)
    v0 = jax.random.normal(key, (N,), dtype=jnp.complex128)
    matvec = lambda v: A @ v

    tn, _, infn = arnoldi_restarted_lm_jit(
        matvec, v0, m=m, max_cycles=max_cycles, tol=tol
    )

    # IRAM with default k (= min(max(2,3), m-1) = 3 for nev=1)
    ti, _, infi = arnoldi_iram_lm_jit(
        matvec, v0, m=m, nev=1, max_cycles=max_cycles, tol=tol
    )

    # Also try k=1 to confirm it matches naive restart
    ti1, _, infi1 = arnoldi_iram_lm_jit(
        matvec, v0, m=m, nev=1, k=10, max_cycles=max_cycles, tol=tol
    )

    k_default = min(max(2, 3), m - 1)
    p_default = m - k_default

    print(f"\n{'=' * 70}")
    print(f"  {name}  (N={N}, m={m}, k_default={k_default}, p={p_default})")
    print(f"{'=' * 70}")
    print(f"  True λ_max  = {true_eval}")
    print(f"  Naive:     λ={tn:.10f}  cycles={infn.n_cycles:3d}  "
          f"matvecs={infn.n_inner_iters:5d}  ritz_res={infn.ritz_residual:.2e}")
    print(f"  IRAM k=1:  λ={ti1:.10f}  cycles={infi1.n_cycles:3d}  "
          f"matvecs={infi1.n_inner_iters:5d}  ritz_res={infi1.ritz_residual:.2e}")
    print(f"  IRAM k={k_default}:  λ={ti:.10f}  cycles={infi.n_cycles:3d}  "
          f"matvecs={infi.n_inner_iters:5d}  ritz_res={infi.ritz_residual:.2e}")
    err_n = jnp.abs(tn - true_eval)
    err_i = jnp.abs(ti - true_eval)
    print(f"  Error: naive={err_n:.2e}  IRAM={err_i:.2e}")


# 1. Diagonal — well-separated eigenvalues
N = 100
test("Diagonal (well-separated)", jnp.diag(jnp.arange(1.0, N + 1)), m=15)

# 2. Random symmetric
key = jax.random.PRNGKey(123)
M = jax.random.normal(key, (200, 200), dtype=jnp.float64)
test("Random symmetric", (M + M.T) / 2, m=20, tol=1e-10)

# 3. Non-symmetric
A = jax.random.normal(jax.random.PRNGKey(999), (150, 150), dtype=jnp.float64)
test("Random non-symmetric", A, m=25, tol=1e-10)

# 4. JIT compilation test
print(f"\n{'=' * 70}")
print("  JIT compilation test")
print(f"{'=' * 70}")
A = jnp.diag(jnp.arange(1.0, 51.0))

@jax.jit
def run(v0):
    return arnoldi_iram_lm_jit(
        lambda v: A @ v, v0, m=10, nev=1, max_cycles=30, tol=1e-10
    )

v0 = jax.random.normal(jax.random.PRNGKey(0), (50,), dtype=jnp.complex128)
t, _, info = run(v0)
print(f"  JIT OK: λ={jnp.real(t):.10f} (expect 50), cycles={info.n_cycles}")


  Diagonal (well-separated)  (N=100, m=15, k_default=3, p=12)
  True λ_max  = (100+0j)
  Naive:     λ=100.0000000000+0.0000000000j  cycles= 14  matvecs=  210  ritz_res=1.12e-13
  IRAM k=1:  λ=100.0000000000-0.0000000000j  cycles= 16  matvecs=   90  ritz_res=2.07e-13
  IRAM k=3:  λ=100.0000000000-0.0000000000j  cycles=  9  matvecs=  111  ritz_res=6.90e-13
  Error: naive=2.29e-16  IRAM=2.27e-13

  Random symmetric  (N=200, m=20, k_default=3, p=17)
  True λ_max  = (19.90850298592405+0j)
  Naive:     λ=19.9085029859+0.0000000000j  cycles=  6  matvecs=  120  ritz_res=4.26e-11
  IRAM k=1:  λ=19.9085029859+0.0000000000j  cycles=  7  matvecs=   80  ritz_res=9.46e-13
  IRAM k=3:  λ=19.9085029859+0.0000000000j  cycles=  5  matvecs=   88  ritz_res=3.05e-12
  Error: naive=1.07e-13  IRAM=5.33e-14

  Random non-symmetric  (N=150, m=25, k_default=3, p=22)
  True λ_max  = (0.8613645250255717+12.490419235958985j)
  Naive:     λ=0.8613645243-12.4904192367j  cycles= 80  matvecs= 2000  ritz_res=3.84e-09


In [ ]:
# Test 3: bypass custom_linear_solve entirely
from dynamiqs.steady_state.core.arnoldi_iram import arnoldi_iram_lm_jit
from dynamiqs.steady_state.api.utils import (
    from_matrix, to_matrix, to_dm, from_dm, update_preconditioner,
)
from dynamiqs.steady_state.preconditionner.lyapunov_solver import LyapunovSolverEig

n = 40
delta = 35.90
H, Ls = build_kerr_oscillator(n, delta)
dims = H.dims
H_jax = H.to_jax()
dtype = H_jax.dtype
Ls_q = dq.stack(Ls)

identity_vec = from_matrix(jnp.eye(n, dtype=dtype))

def kraus_vec(x):
    rho = to_dm(x, n=n, dims=dims)
    return from_matrix((Ls_q @ rho @ Ls_q.dag()).sum(0).to_jax())

LdagL = (Ls_q.dag() @ Ls_q).sum(0).to_jax()
G = 1j * H_jax + 0.5 * LdagL
solver_lyap = LyapunovSolverEig(G)
def precond(x):
    return -from_matrix(solver_lyap.solve(to_matrix(x, n=n), mu=0.0))

def precond_kraus(x):
    return precond(kraus_vec(x))

def stop_metric(vec):
    rho = to_matrix(vec, n=n)
    rho = 0.5 * (rho + rho.conj().mT)
    tr = jnp.trace(rho)
    tr = jnp.where(jnp.abs(tr) > 0, tr, 1.0)
    rho = rho / tr
    rho_q = dq.asqarray(rho, dims=dims)
    return jnp.max(jnp.abs(dq.lindbladian(H, Ls, rho_q).to_jax()))

v0 = from_dm(dq.asqarray(jnp.eye(n, dtype=dtype) / n, dims=dims))

eigval, eigvec, info = arnoldi_iram_lm_jit(
    precond_kraus, v0,
    m=30, nev=1, k=10,
    max_cycles=100, tol=1e-6,
    stop_fn=stop_metric,
)

# Direct hermitise + normalise (NO custom_linear_solve)
rho_direct = to_matrix(eigvec, n=n)
rho_direct = 0.5 * (rho_direct + rho_direct.conj().mT)
rho_direct = rho_direct / jnp.trace(rho_direct)
norm_direct = jnp.max(jnp.abs(dq.lindbladian(H, Ls, dq.asqarray(rho_direct, dims=dims)).to_jax()))

print(f"IRAM direct (no custom_linear_solve): {norm_direct:.6e}")
print(f"IRAM info metric:                     {info.metric_hist_cycles[info.n_cycles-1]:.6e}")
print(f"IRAM ritz residual:                   {info.ritz_residual:.6e}")
print(f"IRAM cycles:                          {info.n_cycles}")

In [ ]:
from systems import build_kerr_oscillator, build_random_single_mode
import jax.numpy as jnp
n = 40
delta = 35.90
deltas = jnp.linspace(-20, 20, 15) * 2 * jnp.pi

H, Ls = build_kerr_oscillator(n, delta)

In [ ]:
"""Benchmark: Jacobian of steady-state loss function.

Compares dense vs GMRES+precond at various krylov sizes, for forward + Jacobian.
Also compares vmap vs lax.map batching.

No catographer dependency. N=30 to keep runtime reasonable.
"""

import gc
import time

import dynamiqs as dq
import jax
import jax.numpy as jnp

dq.set_matmul_precision("highest")
dq.set_precision("double")

N = 40
a = dq.destroy(N)
n_hat_jax = dq.number(N).to_jax()
n = N
twopi = 2 * jnp.pi

# Precomputed operator matrices
a_jax = a.to_jax()
adag_jax = a.dag().to_jax()
adag2a2 = (a.dag() @ a.dag() @ a @ a).to_jax()
adaga = (a.dag() @ a).to_jax()
I_vec = jnp.eye(n, dtype=jnp.complex128).flatten(order="F")

n_detunings = 15
delta_vals = jnp.linspace(-20, 20, n_detunings) * twopi

# Fit parameters: [kappa, kerr, eps]
params_true = jnp.array([14.0 * twopi, -1.0 * twopi, 16.0])


def build_H_and_Ls(params, delta):
    kap, kerr, ep = params
    H = (
        -kerr / 2 * adag2a2
        - delta * adaga
        + 1j * jnp.sqrt(kap) * ep * a_jax
        - 1j * jnp.sqrt(kap) * ep * adag_jax
    )
    L = jnp.sqrt(kap) * a_jax
    return dq.asqarray(H), [dq.asqarray(L)]


# Dense: build superoperator, direct solve
def dense_single(params, delta):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    L_sup = dq.slindbladian(H_q, Ls_q).to_jax()
    L_def = L_sup + jnp.outer(I_vec, I_vec)
    x = jnp.linalg.solve(L_def, I_vec)
    rho = x.reshape((n, n), order="F")
    rho = (rho + rho.conj().T) / 2
    rho /= jnp.trace(rho)
    return jnp.trace(rho @ n_hat_jax).real


# GMRES with Lyapunov preconditioner (dynamiqs)
def gmres_single(params, delta, ks=64):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    solver = dq.SteadyStateArnoldi(krylov_size=ks, tol=1e-6, max_cycles=10,n_refinement=3)
    result = dq.steadystate(H_q, Ls_q, solver=solver)
    return jnp.trace(result.rho.to_jax() @ n_hat_jax).real


# ── Compute dense reference ──────────────────────────────────────
print(f"Kerr oscillator: N={N}, {n_detunings} detunings, 3 fit params")
print("Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]")
print("Computing dense reference...")
ref_fwd_fn = jax.jit(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
n_ref = ref_fwd_fn(params_true).block_until_ready()
ref_jac_fn = jax.jit(
    jax.jacfwd(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
)
jac_ref = ref_jac_fn(params_true).block_until_ready()
print("Done.\n")


def run_benchmark(label, single_fn, krylov_sizes):
    print(f"\n{'=' * 95}")
    print(f"{label}")
    print(f"{'=' * 95}")
    print(
        f"  {'ks':>4}  {'mode':>8}  {'fwd(ms)':>8}  {'jac(ms)':>8}  "
        f"{'total(ms)':>10}  {'fwd_err':>10}  {'jac_err':>10}"
    )
    print(f"  {'-' * 82}")

    for ks in krylov_sizes:
        if ks is None:
            solve_fn = single_fn
        else:
            solve_fn = lambda p, d, _ks=ks: single_fn(p, d, _ks)

        for mode in ["vmap", "lax.map"]:
            jax.clear_caches()
            gc.collect()

            if mode == "vmap":
                batched = lambda p: jax.vmap(solve_fn, in_axes=(None, 0))(p, delta_vals)
            else:
                batched = lambda p: jax.lax.map(lambda d: solve_fn(p, d), delta_vals)

            # Forward
            fn_fwd = jax.jit(batched)
            _ = fn_fwd(params_true).block_until_ready()
            ts = []
            for _ in range(3):
                t0 = time.time()
                vals = fn_fwd(params_true).block_until_ready()
                ts.append(time.time() - t0)
            t_fwd = min(ts) * 1000
            fwd_err = float(jnp.max(jnp.abs(vals - n_ref)))

            # Jacobian
            fn_jac = jax.jit(jax.jacfwd(batched))
            _ = fn_jac(params_true).block_until_ready()
            ts = []
            for _ in range(3):
                t0 = time.time()
                jac = fn_jac(params_true).block_until_ready()
                ts.append(time.time() - t0)
            t_jac = min(ts) * 1000
            jac_err = float(jnp.max(jnp.abs(jac - jac_ref)))
            has_nan = bool(jnp.any(jnp.isnan(jac)))

            ks_str = f"{ks}" if ks is not None else "  —"
            nan_tag = " NaN!" if has_nan else ""
            wrong_tag = " ←WRONG" if fwd_err > 0.1 else ""
            print(
                f"  {ks_str:>4}  {mode:>8}  {t_fwd:8.0f}  {t_jac:8.0f}  "
                f"{t_fwd + t_jac:10.0f}  {fwd_err:10.2e}{wrong_tag}  "
                f"{jac_err:10.2e}{nan_tag}"
            )


# ── Run benchmarks ───────────────────────────────────────────────
#run_benchmark("DENSE SOLVER", dense_single, [None])

run_benchmark(
    "Arnoldi",
    gmres_single,
    [100],
)

Kerr oscillator: N=40, 15 detunings, 3 fit params
Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]
Computing dense reference...
Done.


Arnoldi
    ks      mode   fwd(ms)   jac(ms)   total(ms)     fwd_err     jac_err
  ----------------------------------------------------------------------------------
   100      vmap     16297     22392       38688    1.03e-07    5.46e-07
   100   lax.map      4334      8084       12418    1.09e-07    5.45e-07


In [2]:
"""Benchmark: Jacobian of steady-state loss function.

Compares dense vs GMRES+precond at various krylov sizes, for forward + Jacobian.
Also compares vmap vs lax.map batching.

No catographer dependency. N=30 to keep runtime reasonable.
"""

import gc
import time

import dynamiqs as dq
import jax
import jax.numpy as jnp

dq.set_matmul_precision("highest")
dq.set_precision("double")

N = 40
a = dq.destroy(N)
n_hat_jax = dq.number(N).to_jax()
n = N
twopi = 2 * jnp.pi

# Precomputed operator matrices
a_jax = a.to_jax()
adag_jax = a.dag().to_jax()
adag2a2 = (a.dag() @ a.dag() @ a @ a).to_jax()
adaga = (a.dag() @ a).to_jax()
I_vec = jnp.eye(n, dtype=jnp.complex128).flatten(order="F")

n_detunings = 3
delta_vals = jnp.linspace(-20, 20, n_detunings) * twopi

# Fit parameters: [kappa, kerr, eps]
params_true = jnp.array([14.0 * twopi, -1.0 * twopi, 16.0])


def build_H_and_Ls(params, delta):
    kap, kerr, ep = params
    H = (
        -kerr / 2 * adag2a2
        - delta * adaga
        + 1j * jnp.sqrt(kap) * ep * a_jax
        - 1j * jnp.sqrt(kap) * ep * adag_jax
    )
    L = jnp.sqrt(kap) * a_jax
    return dq.asqarray(H), [dq.asqarray(L)]


# Dense: build superoperator, direct solve
def dense_single(params, delta):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    L_sup = dq.slindbladian(H_q, Ls_q).to_jax()
    L_def = L_sup + jnp.outer(I_vec, I_vec)
    x = jnp.linalg.solve(L_def, I_vec)
    rho = x.reshape((n, n), order="F")
    rho = (rho + rho.conj().T) / 2
    rho /= jnp.trace(rho)
    return jnp.trace(rho @ n_hat_jax).real


# GMRES with Lyapunov preconditioner (dynamiqs)
def gmres_single(params, delta, ks=64, n_refinement=3):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    solver = dq.SteadyStateGMRES(krylov_size=ks, tol=1e-6, max_iteration=10, n_refinement=n_refinement)
    result = dq.steadystate(H_q, Ls_q, solver=solver)
    return jnp.trace(result.rho.to_jax() @ n_hat_jax).real


# ── Compute dense reference ──────────────────────────────────────
print(f"Kerr oscillator: N={N}, {n_detunings} detunings, 3 fit params")
print("Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]")
print("Computing dense reference...")
ref_fwd_fn = jax.jit(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
n_ref = ref_fwd_fn(params_true).block_until_ready()
ref_jac_fn = jax.jit(
    jax.jacfwd(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
)
jac_ref = ref_jac_fn(params_true).block_until_ready()
print("Done.\n")


def run_benchmark(label, single_fn, krylov_sizes,n_refinement):
    print(f"\n{'=' * 95}")
    print(f"{label}")
    print(f"{'=' * 95}")
    print(
        f"  {'ks':>4}  {'mode':>8}  {'fwd(ms)':>8}  {'jac(ms)':>8}  "
        f"{'total(ms)':>10}  {'fwd_err':>10}  {'jac_err':>10}"
    )
    print(f"  {'-' * 82}")

    for n_ref in n_refinement:
        for ks in krylov_sizes:
            if ks is None:
                solve_fn = single_fn
            else:
                solve_fn = lambda p, d, _ks=ks, _n_ref=n_ref: single_fn(p, d, _ks, _n_ref)

            for mode in ["vmap", "lax.map"]:
                jax.clear_caches()
                gc.collect()

                if mode == "vmap":
                    batched = lambda p: jax.vmap(solve_fn, in_axes=(None, 0))(p, delta_vals)
                else:
                    batched = lambda p: jax.lax.map(lambda d: solve_fn(p, d), delta_vals)

                # Forward
                fn_fwd = jax.jit(batched)
                _ = fn_fwd(params_true).block_until_ready()
                ts = []
                for _ in range(3):
                    t0 = time.time()
                    vals = fn_fwd(params_true).block_until_ready()
                    ts.append(time.time() - t0)
                t_fwd = min(ts) * 1000
                fwd_err = float(jnp.max(jnp.abs(vals - n_ref)))

                # Jacobian
                fn_jac = jax.jit(jax.jacfwd(batched))
                _ = fn_jac(params_true).block_until_ready()
                ts = []
                for _ in range(3):
                    t0 = time.time()
                    jac = fn_jac(params_true).block_until_ready()
                    ts.append(time.time() - t0)
                t_jac = min(ts) * 1000
                jac_err = float(jnp.max(jnp.abs(jac - jac_ref)))
                has_nan = bool(jnp.any(jnp.isnan(jac)))

                ks_str = f"{ks}" if ks is not None else "  —"
                nan_tag = " NaN!" if has_nan else ""
                wrong_tag = " ←WRONG" if fwd_err > 0.1 else ""
                print(
                    f"  {ks_str:>4}  {mode:>8}  {t_fwd:8.0f}  {t_jac:8.0f}  "
                    f"{t_fwd + t_jac:10.0f}  {fwd_err:10.2e}{wrong_tag}  "
                    f"{jac_err:10.2e}{nan_tag}"
                )


# ── Run benchmarks ───────────────────────────────────────────────
#run_benchmark("DENSE SOLVER", dense_single, [None])

run_benchmark(
    "GMRES",
    gmres_single,
    [100],[0,3]
)

Kerr oscillator: N=40, 3 detunings, 3 fit params
Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]
Computing dense reference...
Done.


GMRES
    ks      mode   fwd(ms)   jac(ms)   total(ms)     fwd_err     jac_err
  ----------------------------------------------------------------------------------


TypeError: SteadyStateGMRES.__init__() got an unexpected keyword argument 'max_iteration'